In [1]:
import numpy as np
import heapq
import pandas as pd
from scipy.stats import skewnorm


In [2]:

# =====================================================
# PARAMETERS
# =====================================================

SIM_TIME             = 45 * 24
DRIVER_RATE          = 4.74
RIDER_RATE           = 34.6
PATIENCE_RATE        = 5
SPEED                = 20
AREA_SIZE            = 20
BASE_FARE            = 3
FARE_PER_MILE        = 2
PETROL_COST_PER_MILE = 0.20
CVAR_ALPHA           = 0.05

# HUB_WEIGHT < 1  → hub more attractive (driver prefers hub)
# HUB_WEIGHT = 1  → pure distance comparison
# HUB_WEIGHT > 1  → rider must be closer than hub to win
# Rule: go to hub if  HUB_WEIGHT * hub_dist < nearest_rider_dist
HUB_WEIGHT = 1.5

# 5 hubs: Centre + North + South + East + West
# Placed at mean pickup density per zone from real BoxCar data
HUBS = [
    np.array([ 9.69, 10.40]),   # Centre
    np.array([ 8.98, 16.55]),   # North
    np.array([ 7.28,  4.09]),   # South
    np.array([16.05, 13.40]),   # East
    np.array([ 3.60, 11.69]),   # West
]


In [3]:

# =====================================================
# LOCATION FUNCTIONS
# =====================================================

def driver_initial_location():
    x = np.clip(np.random.normal(9.97,  4.36), 0, AREA_SIZE)
    y = np.clip(np.random.normal(11.51, 4.34), 0, AREA_SIZE)
    return np.array([x, y])

def rider_pickup_location():
    x = np.clip(skewnorm.rvs(a= 1.80, loc= 4.18, scale=5.97), 0, AREA_SIZE)
    y = np.clip(skewnorm.rvs(a=-2.49, loc=17.05, scale=6.31), 0, AREA_SIZE)
    return np.array([x, y])

def rider_dropoff_location():
    x = np.clip(skewnorm.rvs(a= -1.65, loc=15.51, scale=6.24), 0, AREA_SIZE)
    y = np.clip(skewnorm.rvs(a=-14.26, loc=19.50, scale=7.50), 0, AREA_SIZE)
    return np.array([x, y])

def nearest_hub(location):
    return min(HUBS, key=lambda h: np.linalg.norm(location - h))


In [4]:

# =====================================================
# DRIVER & RIDER CLASSES
# =====================================================

class Driver:
    def __init__(self, driver_id, location, online_until):
        self.id              = driver_id
        self.location        = location
        self.online_until    = online_until
        self.earnings        = 0
        self.status          = "idle"
        self.busy_time       = 0
        self.relocation_cost = 0

class Rider:
    def __init__(self, rider_id, origin, destination, arrival_time, abandon_time):
        self.id           = rider_id
        self.origin       = origin
        self.destination  = destination
        self.arrival_time = arrival_time
        self.abandon_time = abandon_time
        self.matched      = False


In [5]:

# =====================================================
# SIMULATION
# =====================================================

class Simulation:

    def __init__(self, use_hubs=False, hub_weight=HUB_WEIGHT):
        self.use_hubs   = use_hubs
        self.hub_weight = hub_weight

        self.current_time       = 0
        self.event_list         = []
        self.idle_drivers       = {}
        self.busy_drivers       = {}
        self.relocating_drivers = {}
        self.waiting_riders     = {}

        self.driver_counter   = 0
        self.rider_counter    = 0
        self.total_riders     = 0
        self.abandoned_riders = 0
        self.total_wait_time  = 0
        self.driver_log       = []

    # -------------------------------------------------
    def exp_time(self, rate):
        return np.random.exponential(1 / rate)

    def distance(self, a, b):
        return np.linalg.norm(np.array(a) - np.array(b))

    def travel_time(self, dist):
        mu = dist / SPEED
        return np.random.uniform(0.8 * mu, 1.2 * mu)

    def schedule_event(self, time, event_type, data=None):
        heapq.heappush(self.event_list, (time, event_type, data))

    # =====================================================
    # STRATEGIC DECISION
    # Called immediately when driver becomes idle.
    # Compares weighted hub distance vs nearest rider.
    # Goes to hub if:  hub_weight * hub_dist < rider_dist
    # =====================================================

    def driver_decision(self, driver_id):
        if driver_id not in self.idle_drivers:
            return

        driver = self.idle_drivers[driver_id]
        if self.current_time >= driver.online_until:
            return

        hub      = nearest_hub(driver.location)
        hub_dist = self.distance(driver.location, hub)

        # Already at a hub — no need to move
        if hub_dist < 0.5:
            return

        # Distance to nearest waiting rider (inf if none)
        if self.waiting_riders:
            rider_dist = min(
                self.distance(driver.location, r.origin)
                for r in self.waiting_riders.values()
            )
        else:
            rider_dist = np.inf

        # Go to hub only if weighted hub distance beats nearest rider
        if self.hub_weight * hub_dist < rider_dist:
            reloc_cost = PETROL_COST_PER_MILE * hub_dist

            driver.earnings        -= reloc_cost
            driver.relocation_cost += reloc_cost
            driver.status           = "relocating"

            for d in self.driver_log:
                if d["driver_id"] == driver_id:
                    d["earnings"]        -= reloc_cost
                    d["relocation_cost"] += reloc_cost
                    d["n_relocations"]   += 1
                    break

            self.relocating_drivers[driver_id] = driver
            del self.idle_drivers[driver_id]

            self.schedule_event(
                self.current_time + self.travel_time(hub_dist),
                "DRIVER_AT_HUB",
                (driver_id, hub.copy())
            )

    # =====================================================
    # MATCHING
    # =====================================================

    def try_match(self):
        if not self.idle_drivers or not self.waiting_riders:
            return

        driver_id = next(iter(self.idle_drivers))
        driver    = self.idle_drivers[driver_id]

        rider_id = min(
            self.waiting_riders,
            key=lambda r: self.distance(driver.location,
                                        self.waiting_riders[r].origin)
        )

        rider         = self.waiting_riders[rider_id]
        rider.matched = True
        self.total_wait_time += self.current_time - rider.arrival_time

        pickup_dist = self.distance(driver.location, rider.origin)
        trip_dist   = self.distance(rider.origin, rider.destination)
        pickup_time = self.travel_time(pickup_dist)
        trip_time   = self.travel_time(trip_dist)

        payment  = BASE_FARE + FARE_PER_MILE * trip_dist
        petrol   = PETROL_COST_PER_MILE * (pickup_dist + trip_dist)
        earnings = payment - petrol

        driver.status     = "busy"
        driver.busy_time += pickup_time + trip_time

        self.busy_drivers[driver_id] = driver
        del self.idle_drivers[driver_id]
        del self.waiting_riders[rider_id]

        self.schedule_event(
            self.current_time + pickup_time + trip_time,
            "DROPOFF",
            (driver_id, rider.destination, earnings)
        )

    # =====================================================
    # EVENT HANDLERS
    # =====================================================

    def handle_driver_arrival(self):
        self.driver_counter += 1
        driver_id = self.driver_counter

        location        = driver_initial_location()
        online_duration = np.random.uniform(6, 8)
        online_until    = self.current_time + online_duration

        driver = Driver(driver_id, location, online_until)
        self.idle_drivers[driver_id] = driver

        self.driver_log.append({
            "driver_id"      : driver_id,
            "arrival_time"   : self.current_time,
            "online_until"   : online_until,
            "earnings"       : 0,
            "busy_time"      : 0,
            "relocation_cost": 0,
            "n_relocations"  : 0
        })

        self.schedule_event(online_until, "DRIVER_OFFLINE", driver_id)
        self.schedule_event(
            self.current_time + self.exp_time(DRIVER_RATE), "DRIVER_ARRIVAL"
        )

        # FIX 1: decision BEFORE try_match — driver chooses hub before
        # being assigned to any rider
        if self.use_hubs:
            self.driver_decision(driver_id)

        # Only match if driver didn't just leave for hub
        if driver_id in self.idle_drivers:
            self.try_match()

    def handle_dropoff(self, data):
        driver_id, new_location, earnings = data
        driver = self.busy_drivers[driver_id]

        driver.earnings += earnings
        driver.location  = new_location
        driver.status    = "idle"

        for d in self.driver_log:
            if d["driver_id"] == driver_id:
                d["earnings"]  += earnings
                d["busy_time"]  = driver.busy_time
                break

        del self.busy_drivers[driver_id]

        if self.current_time < driver.online_until:
            self.idle_drivers[driver_id] = driver

            # FIX 1 again: decision before match after dropoff too
            if self.use_hubs:
                self.driver_decision(driver_id)

        if driver_id in self.idle_drivers:
            self.try_match()

    def handle_driver_at_hub(self, data):
        driver_id, hub_location = data

        if driver_id not in self.relocating_drivers:
            return

        driver          = self.relocating_drivers[driver_id]
        driver.location = hub_location
        driver.status   = "idle"
        del self.relocating_drivers[driver_id]

        if self.current_time < driver.online_until:
            self.idle_drivers[driver_id] = driver

        # FIX 2: re-evaluate at hub — if no riders appeared, stay;
        # if riders arrived during relocation, match immediately
        # hub_dist < 0.5 check in driver_decision prevents re-relocation
        self.try_match()

    # =====================================================
    # RUN
    # =====================================================

    def run(self):
        self.schedule_event(0, "DRIVER_ARRIVAL")
        self.schedule_event(0, "RIDER_ARRIVAL")

        while self.event_list and self.current_time < SIM_TIME:
            self.current_time, event_type, data = heapq.heappop(self.event_list)

            if event_type == "DRIVER_ARRIVAL":
                self.handle_driver_arrival()

            elif event_type == "RIDER_ARRIVAL":
                self.rider_counter += 1
                rider_id = self.rider_counter
                self.total_riders += 1

                origin       = rider_pickup_location()
                destination  = rider_dropoff_location()
                abandon_time = self.current_time + self.exp_time(PATIENCE_RATE)

                rider = Rider(rider_id, origin, destination,
                              self.current_time, abandon_time)
                self.waiting_riders[rider_id] = rider

                self.schedule_event(abandon_time, "RIDER_ABANDON", rider_id)
                self.schedule_event(
                    self.current_time + self.exp_time(RIDER_RATE), "RIDER_ARRIVAL"
                )
                self.try_match()

            elif event_type == "DROPOFF":
                self.handle_dropoff(data)

            elif event_type == "RIDER_ABANDON":
                if data in self.waiting_riders:
                    self.abandoned_riders += 1
                    del self.waiting_riders[data]

            elif event_type == "DRIVER_OFFLINE":
                self.idle_drivers.pop(data, None)
                self.relocating_drivers.pop(data, None)

            elif event_type == "DRIVER_AT_HUB":
                self.handle_driver_at_hub(data)

        return self.collect_results()

    # =====================================================
    # RESULTS
    # =====================================================

    def collect_results(self):
        matched          = self.total_riders - self.abandoned_riders
        abandonment_rate = self.abandoned_riders / self.total_riders
        avg_wait         = self.total_wait_time / matched if matched > 0 else 0

        driver_df = pd.DataFrame(self.driver_log)
        driver_df["online_duration"]   = driver_df["online_until"] - driver_df["arrival_time"]
        driver_df["busy_time_capped"]  = driver_df[["busy_time","online_duration"]].min(axis=1)
        driver_df["idle_time"]         = driver_df["online_duration"] - driver_df["busy_time_capped"]

        profits     = driver_df["earnings"].values
        N           = len(profits)
        mean_profit = profits.mean()
        std_profit  = profits.std(ddof=1)
        fairness_cv = std_profit / mean_profit if mean_profit > 0 else 0

        profits_sorted = np.sort(profits)
        cutoff         = max(1, int(np.floor(CVAR_ALPHA * N)))
        cvar           = np.mean(profits_sorted[:cutoff])
        cvar_median    = np.mean(profits_sorted[:max(1, int(np.floor(0.5 * N)))])

        return {
            "abandonment_rate" : abandonment_rate,
            "avg_wait"         : avg_wait,
            "matched_riders"   : matched,
            "total_riders"     : self.total_riders,
            "abandoned_riders" : self.abandoned_riders,
            "total_drivers"    : N,
            "avg_earnings"     : mean_profit,
            "avg_profit_per_hr": driver_df["earnings"].sum() / driver_df["online_duration"].sum(),
            "avg_idle_time"    : driver_df["idle_time"].mean(),
            "fairness_cv"      : fairness_cv,
            "cvar"             : cvar,
            "delta_cvar"       : cvar - cvar_median,
            "total_reloc_cost" : driver_df["relocation_cost"].sum(),
            "avg_relocations"  : driver_df["n_relocations"].mean(),
        }


In [6]:

# =====================================================
# RUN COMPARISON — Baseline vs multiple weights
# =====================================================

weights = [1.2, 1.5, 2.0]
results = {}

np.random.seed(42)
print("Running Baseline (no hubs)...")
results["baseline"] = Simulation(use_hubs=False).run()

for w in weights:
    np.random.seed(42)
    print(f"Running 5-hub strategic, weight={w}...")
    results[w] = Simulation(use_hubs=True, hub_weight=w).run()

# ─── Print table ──────────────────────────────────────────
b = results["baseline"]
print("\n" + "="*88)
print(f"  {'Config':<18} {'Abandon%':>9} {'Wait(min)':>10} {'AvgEarn£':>9} "
      f"{'£/hr':>7} {'CV':>7} {'CVaR£':>8} {'RelocCost£':>11} {'Relocs':>7}")
print("-"*88)
for label, key in [("Baseline", "baseline")] + [(f"5-hub w={w}", w) for w in weights]:
    r = results[key]
    flag = ""
    if key != "baseline":
        diff = r['abandonment_rate'] - b['abandonment_rate']
        flag = f"({'↓' if diff<0 else '↑'}{abs(diff)*100:.2f}pp)"
    print(f"  {label:<18} {r['abandonment_rate']*100:>8.2f}% "
          f"{r['avg_wait']*60:>10.3f} "
          f"{r['avg_earnings']:>9.2f} "
          f"{r['avg_profit_per_hr']:>7.2f} "
          f"{r['fairness_cv']:>7.4f} "
          f"{r['cvar']:>8.2f} "
          f"{r['total_reloc_cost']:>11.2f} "
          f"{r['avg_relocations']:>7.2f}  {flag}")
print("="*88)

Running Baseline (no hubs)...
Running 5-hub strategic, weight=1.2...
Running 5-hub strategic, weight=1.5...
Running 5-hub strategic, weight=2.0...

  Config              Abandon%  Wait(min)  AvgEarn£    £/hr      CV    CVaR£  RelocCost£  Relocs
----------------------------------------------------------------------------------------
  Baseline               3.44%      0.306    111.77   15.96  0.1995    66.95        0.00    0.00  
  5-hub w=1.2            6.00%      0.555    106.40   15.21  0.1958    63.92    22330.54    6.67  (↑2.56pp)
  5-hub w=1.5            5.53%      0.546    107.68   15.38  0.1928    65.15    21880.10    6.59  (↑2.09pp)
  5-hub w=2.0            5.84%      0.565    110.54   15.78  0.1953    66.20    20980.20    6.47  (↑2.40pp)
